## Krish naik 3.0 agentic AI notes

####  uv add langchain-mcp-adapters [if langchain 1.3.x]
#### uv add "langchain[mcp]"  [if langchain 1.4.x]
#### we are using 1.4.x approach of package here
#### uv add fastmcp if runnign in memory mcp server

In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.mcp import MCPAdapter
from langchain_core.messages import ToolMessage

from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport


/var/folders/w7/bjnp92015253vpp0xtyzp16c0000gn/T/ipykernel_82691/3698944834.py:5: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [2]:
load_dotenv()

True

In [5]:
agent = create_agent(
    "openai:gpt-5-mini"
)
response = agent.invoke({ "messages":[
    HumanMessage('Hi')
]})
print(response["messages"][-1].content)

Hi — how can I help you today?


## MCP server accessed as STDIO using langchain-mcp

In [3]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("../MCP in Langchain/cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


## without explict stdio transport

In [ ]:
from pathlib import Path
from langchain.mcp import MCPAdapter

stdio_adapter = MCPAdapter(Path("../MCP in Langchain/cinebot_mcp_server.py"))

tools=await stdio_adapter.list_tools()
print("Available tools:", [t.name for t in tools])

## invoking tool after its received from mcp server - without / with agent

In [13]:
tool_call_result=''
async with MCPAdapter(cinebot_mcp_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    # since we are using async mcp client - mcp adapter with async we need to use await and ainvoke
    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well

    try:
        cinebot_agent = create_agent(model='openai:gpt-5-mini',tools=tools)
        result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        tool_call_result = result
        print("Agent invocation result:")
        from rich import print
        print(result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_640031cb-ef96-4de3-80bb-e5a81473ea30'}]
Agent invocation result:


{
    'messages': [
        HumanMessage(
            content='What are the showtimes for Interstellar?',
            additional_kwargs={},
            response_metadata={},
            id='56425f26-c9c2-45a0-a56d-c9600401068b'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 137,
                    'total_tokens': 227,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUux7s9gr8ACWZOEfodJhrDcqIym2',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10226-ff6a-7960-b463-c0409973e207-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'call_Ikef3igbqJO1SfqY0mClyXtZ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 137,
                'output_tokens': 90,
                'total_tokens': 227,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content=[
                {'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_60774dbe-c631-4bc0-888c-85e87910d45b'}
            ],
            name='check_showtimes',
            id='62701660-ab02-45b3-9433-6cea7b046daa',
            tool_call_id='call_Ikef3igbqJO1SfqY0mClyXtZ',
            artifact={'structured_content': {'result': '7:00 PM and 10:15 PM'}}
        ),
        AIMessage(
            content='The available showtimes for Interstellar are 7:00 PM and 10:15 PM.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 179,
                    'total_tokens': 202,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUux90ixbDfOvYWSMN9eWVVuesVoj',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a10227-0835-7403-9cbf-45fb57fd8b11-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 179,
                'output_tokens': 23,
                'total_tokens': 202,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

## Invoking the remote HTTP MCP Server

In [ ]:
async def main(message):
    async with MCPAdapter( "https://time-tracker-livid-theta.vercel.app/mcp") as adapter:
        tools = await adapter.list_tools()
        agent = create_agent( "openai:gpt-5-mini", tools= tools)
        print(message)
        return await agent.ainvoke({ "messages":[HumanMessage(message)]})

In [ ]:
result = await main('Hi')
print(result["messages"][-1].content)

In [ ]:
result = await main('what projects do you track?  ')
print(result["messages"][-1].content)

In [ ]:
result = await main(' can you log 10 hours against my account amohan7ai-v3 for 2026-09-27 for project "langchain with mcp adapter". share updated project list after that  ')
print(result["messages"][-1].content)

## Invoking inmemory MCP server

In [ ]:
from fastmcp import FastMCP
mcp_inmemory = FastMCP()
@mcp_inmemory.tool
def add(a,b):
    print(f'in memory mcp adding {a} and {b}')
    return a+b

In [ ]:
async def mcp_inmemory_func(message):
    async with MCPAdapter(mcp_inmemory) as adapter:
        tools = await adapter.list_tools()
        print([("name-",t.name, " arguments=",t.args_schema["properties"]) for t in tools])
        agent = create_agent( "openai:gpt-5-mini", tools= tools)
        print(message)
        return await agent.ainvoke({ "messages":[HumanMessage(message)]})

result = await mcp_inmemory_func('add 2 and 3 ')
print(result["messages"][-1].content)

## Structured output

In [12]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    # message = await seat_map_tool.ainvoke({'movie_title': 'Interstellar'}) > gives raw tool call output
    print("Text Content (what my model reads)",message.content)
    print("Structured Data (what my code can use if required)",message.artifact)
    print(message.artifact['structured_content']['available_rows'])  # Accessing structured data directly

Text Content (what my model reads) [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_7142bb1c-1f09-41af-8a40-ee3ba302facc'}]
Structured Data (what my code can use if required) {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
['A', 'B', 'C']


In [20]:
## structured output
from fastmcp import FastMCP
import random 
from pydantic import BaseModel,Field
from langchain_core.messages import ToolMessage

mcp_inmemory = FastMCP()
@mcp_inmemory.tool
def reserve_seats(name: str, seats: int) -> dict:
    return {
        "name": name,
        "seats": seats,
        "id": f"R {random.randint(1,100)}"
        }

class Reservation(BaseModel):
    name: str = Field("Name of person booked")
    seats: int =Field("number of seats booked")
    id: str = Field("Booking ID")

    
async def reserve_seats_agent(message):
    async with MCPAdapter(mcp_inmemory) as adapter:
        tools = await adapter.list_tools()
        agent = create_agent( "openai:gpt-5-mini", tools= tools, response_format = Reservation)
        print(message)
        return await agent.ainvoke({ "messages":[HumanMessage(message)]})

result = await reserve_seats_agent('book one seat for aj123')
from rich import print
print(result)
for res in result["messages"]:
    if isinstance(res, ToolMessage):
        print(res.artifact)


book one seat for aj123


{
    'messages': [
        HumanMessage(
            content='book one seat for aj123',
            additional_kwargs={},
            response_metadata={},
            id='c701d8c0-37c1-4166-9d41-25bb058443de'
        ),
        AIMessage(
            content='',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 94,
                    'prompt_tokens': 176,
                    'total_tokens': 270,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUvy9eAO87EKDF5OavVZQ1jGVQQUu',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10262-9fba-7630-a9cf-6988b95e21f6-0',
            tool_calls=[
                {
                    'name': 'reserve_seats',
                    'args': {'name': 'aj123', 'seats': 1},
                    'id': 'call_lA9bBiLzoCpaLK1wmoe2WVYY',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 176,
                'output_tokens': 94,
                'total_tokens': 270,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content=[
                {
                    'type': 'text',
                    'text': '{"name":"aj123","seats":1,"id":"R 33"}',
                    'id': 'lc_0efa3407-97fb-494d-b7be-85299fb77eb7'
                }
            ],
            name='reserve_seats',
            id='0d6773e1-81b0-484e-8994-725a48c66f2e',
            tool_call_id='call_lA9bBiLzoCpaLK1wmoe2WVYY',
            artifact={'structured_content': {'name': 'aj123', 'seats': 1, 'id': 'R 33'}}
        ),
        AIMessage(
            content='{"name":"aj123","seats":1,"id":"R 33"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 229,
                    'total_tokens': 252,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUvyBebfXUrIsmMFvDx6qp1aLVbkZ',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a10262-a89a-73a0-977c-a07ce7ba5cbf-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 229,
                'output_tokens': 23,
                'total_tokens': 252,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],

{'structured_content': {'name': 'aj123', 'seats': 1, 'id': 'R 33'}}

## error message

In [13]:
# @mcp.tool()
# def risky_lookup(booking_id: str) -> str:
#     """Look up a booking -- fails if the ID format is wrong."""
#     if not booking_id.startswith("BK"):
#         raise ValueError(f"Invalid booking ID format: {booking_id!r}. Expected it to start with 'BK'.")
#     return f"Booking {booking_id}: confirmed."

error_message_global = ''
error_transport = PythonStdioTransport(
    script_path=Path('../MCP in Langchain/cinebot_error_server.py'),
    log_file=Path('cinebot_error_server.log')
)

async with MCPAdapter(error_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    risky_lookup_tool = tools[0]
    print(risky_lookup_tool)
    bad_call = {'name': 'risky_lookup', 'args': {'booking_id': '12345'}, 'id': 'bad_call_001', 'type': 'tool_call'}

    from rich import print
    error_message = await risky_lookup_tool.ainvoke(bad_call)
    print("Error message from tool :")
    print(error_message)
    print("Content:", error_message.content)
    error_message_global = error_message
    print(error_message)

    
    error_message = await risky_lookup_tool.ainvoke({'booking_id': '12345'})
    print("Error message from tool with tool as input:")
    print(error_message)



    try:
        cinebot_error_agent = create_agent(model='openai:gpt-5-mini',tools=tools)
        result = await cinebot_error_agent.ainvoke({"messages":[('user', "book for 123456")]})
        tool_call_result = result
        print("Error Agent invocation result:")
        from rich import print
        print(result)
    except Exception as e:
        print("Error during agent invocation:", e)

StructuredTool(
    name='risky_lookup',
    description='Look up a booking -- fails if the ID format is wrong.',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'booking_id': {'type': 'string'}},
        'required': ['booking_id']
    },
    metadata={
        'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x10eb10d60>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x12130ca40>
)

Error message from tool :

ToolMessage(
    content=[
        {
            'type': 'text',
            'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start 
with 'BK'.",
            'id': 'lc_1a38a144-c52d-4dc6-9f68-10be74247cd7'
        }
    ],
    name='risky_lookup',
    tool_call_id='bad_call_001',
    status='error'
)

Content:
[
    {
        'type': 'text',
        'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 
'BK'.",
        'id': 'lc_1a38a144-c52d-4dc6-9f68-10be74247cd7'
    }
]

ToolMessage(
    content=[
        {
            'type': 'text',
            'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start 
with 'BK'.",
            'id': 'lc_1a38a144-c52d-4dc6-9f68-10be74247cd7'
        }
    ],
    name='risky_lookup',
    tool_call_id='bad_call_001',
    status='error'
)

Error message from tool with tool as input:

[
    {
        'type': 'text',
        'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 
'BK'.",
        'id': 'lc_77aed0f1-b037-402e-be1d-353add177a8a'
    }
]

Error Agent invocation result:

{
    'messages': [
        HumanMessage(
            content='book for 123456',
            additional_kwargs={},
            response_metadata={},
            id='d4f5ebfd-c370-4270-b978-9a15fd3f74d9'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 137,
                    'total_tokens': 227,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUy4YgzpOtxRecQkDeXRZOYFyc6wv',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a102dd-fed9-7af1-bc8b-d5d9d145b479-0',
            tool_calls=[
                {
                    'name': 'risky_lookup',
                    'args': {'booking_id': '123456'},
                    'id': 'call_954JId2Z9YVTtHs2sO1J5BVn',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 137,
                'output_tokens': 90,
                'total_tokens': 227,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content=[
                {
                    'type': 'text',
                    'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '123456'. Expected it to
start with 'BK'.",
                    'id': 'lc_e6ae2113-1a05-4daa-9713-3f9c216e2271'
                }
            ],
            name='risky_lookup',
            id='1f013cbe-1d67-46ec-ae08-3b14803303e0',
            tool_call_id='call_954JId2Z9YVTtHs2sO1J5BVn',
            status='error'
        ),
        AIMessage(
            content='I couldn’t find a booking with ID "123456" — our booking IDs must start with "BK" (example: 
BK123456). Do you want me to:\n\n1) Look up BK123456 instead (I can try that now) — reply “lookup BK123456” or 
confirm; or  \n2) Make a new booking — if so, tell me the details below.\n\nIf you want a new booking, please 
provide:\n- Service/location (what you’re booking)  \n- Date and time (or date range; include time zone if not 
local)  \n- Number of people/participants  \n- Full name and phone or email for contact  \n- Any special requests 
or accessibility needs  \n- Payment method (if required)\n\nTell me which option and the details you want, and I’ll
proceed.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 492,
                    'prompt_tokens': 194,
                    'total_tokens': 686,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUy4bukvVINI

## Human in the loop middleware for confirmation of tool call

In [11]:
from fastmcp.client.transports import PythonStdioTransport
from pathlib import Path
from langchain.mcp import MCPAdapter
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.messages import HumanMessage

cinebot_transport = PythonStdioTransport(
    script_path=Path("../MCP in Langchain/cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

def is_destructive_tool_call(tool):
    # return message.type == "tool_call" and message.name == "cancel_booking"
    meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
    return bool(meta.get('destructive_hint'))


async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()
    cancel_booking_tool = next(t for t in tools if t.name == "cancel_booking")
    print("cancel_booking_tool ")
    from rich import print
    print(cancel_booking_tool)
    for tool in tools: 
        meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
        has_destructive_hint =  bool(meta.get('destructive_hint'))
        print(f" tool: {tool.name}, has_destructive_hint: {has_destructive_hint}")
    
    interrupt_on = {
        t.name: {"allowed_decisions": ["approve", "edit", "reject"]}
        for t in tools
        if is_destructive_tool_call(t)
    }
    print(interrupt_on)
    hitl_agent = create_agent(model="openai:gpt-5-mini", tools = tools, 
                              middleware = [HumanInTheLoopMiddleware(interrupt_on=interrupt_on)])
    result = hitl_agent.invoke({"messages":HumanMessage("cancel booking for 1234")})
    print(result)

cancel_booking_tool

StructuredTool(
    name='cancel_booking',
    description='Cancel an existing booking. Irreversible.',
    args_schema={
        'type': 'object',
        'additionalProperties': False,
        'properties': {'booking_id': {'type': 'string'}},
        'required': ['booking_id']
    },
    metadata={
        'mcp': {
            'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}},
            'server': {'name': 'CineBot', 'version': '4.0.10'}
        }
    },
    handle_tool_error=<function _handle_mcp_tool_error at 0x11aa50ae0>,
    response_format='content_and_artifact',
    coroutine=<function as_langchain_tool.<locals>.call_tool at 0x134f865c0>
)

tool: check_showtimes, has_destructive_hint: False

tool: cancel_booking, has_destructive_hint: True

tool: get_seat_map, has_destructive_hint: False

{'cancel_booking': {'allowed_decisions': ['approve', 'edit', 'reject']}}

{
    'messages': [
        HumanMessage(
            content='cancel booking for 1234',
            additional_kwargs={},
            response_metadata={},
            id='a33beec6-16e8-4108-8357-51ac1f9ac33c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 25,
                    'prompt_tokens': 191,
                    'total_tokens': 216,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EUyfMju5dhTe12bcJZnr7I1MS9TDl',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10300-d3ba-77c3-8bb6-5ff575f19a78-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': '1234'},
                    'id': 'call_Fg2aU4pHogUO7L1BS0XlGy9G',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 191,
                'output_tokens': 25,
                'total_tokens': 216,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': '1234'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': '1234'}"
                    }
                ],
                'review_configs': [
                    {'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject']}
                ]
            },
            id='bd4f7d07e738449aff9baa560843b210'
        )
    ]
}

## Elicitation with new MCP protocol

In [44]:
from typing import Annotated
from pydantic import BaseModel, Field
from mcp.server.mcpserver import Elicit, Resolve, MCPServer

mcp = MCPServer("CineBot")


class CancelConfirmation(BaseModel):
    confirm: bool = Field(description="Confirm that you want to cancel this booking.")


async def ask_cancel_confirmation(booking_id: str) -> Elicit[CancelConfirmation]:
    print("inside ask_cancel_confirmation")
    return Elicit(f"Are you sure you want to cancel booking {booking_id}?", CancelConfirmation)


@mcp.tool()
async def cancel_booking(booking_id: str, confirmation: Annotated[CancelConfirmation, Resolve(ask_cancel_confirmation)]) -> str:
    if not confirmation.confirm:
        return f"Booking {booking_id} was not cancelled."
    return f"Booking {booking_id} has been cancelled."


from langchain.mcp import MCPAdapter
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from rich import print


async with MCPAdapter(mcp) as adapter:
    tools = await adapter.list_tools()

    agent = create_agent("openai:gpt-5-mini", tools=tools, checkpointer=InMemorySaver())

    config = {"configurable": {"thread_id": "cinebot-123"}}

    result = await agent.ainvoke({"messages": [{"role": "user", "content": "Cancel booking BK12345"}]}, config)

    print(result)

    request_key = result["__interrupt__"][0].value["requests"][0]["key"]
    print("Request key:", request_key)

    result = await agent.ainvoke(Command(resume={"responses": {request_key: {"action": "accept", "content": {"confirm": True}}}}), config)
    print(request_key)
    print(result)

[10/03/26 16:06:13] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=1739411;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1739412;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

inside ask_cancel_confirmation

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK12345',
            additional_kwargs={},
            response_metadata={},
            id='76de4287-d692-42da-ad17-e278fe59104e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 123,
                    'total_tokens': 213,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EV0HkchN3tGB3pBLRNUdpsIUNoQeo',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1035f-ca30-7f30-88c3-f34515adfa2e-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK12345'},
                    'id': 'call_PpRt65mwTxedRLz8ZMuGaaVq',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 123,
                'output_tokens': 90,
                'total_tokens': 213,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'type': 'mcp_elicitation',
                'tool_name': 'cancel_booking',
                'requests': [
                    {
                        'key': '__main__:ask_cancel_confirmation',
                        'message': 'Are you sure you want to cancel booking BK12345?',
                        'mode': 'form',
                        'requested_schema': {
                            'properties': {
                                'confirm': {
                                    'description': 'Confirm that you want to cancel this booking.',
                                    'title': 'Confirm',
                                    'type': 'boolean'
                                }
                            },
                            'required': ['confirm'],
                            'type': 'object'
                        }
                    }
                ]
            },
            id='85a2fcd16b9f1e3a10c47b5417f178fe'
        )
    ]
}

Request key: __main__:ask_cancel_confirmation

inside ask_cancel_confirmation

inside ask_cancel_confirmation

[10/03/26 16:06:15] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=1739417;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=1739418;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             "HTTP/1.1 200 OK"                                                                     

__main__:ask_cancel_confirmation

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK12345',
            additional_kwargs={},
            response_metadata={},
            id='76de4287-d692-42da-ad17-e278fe59104e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 90,
                    'prompt_tokens': 123,
                    'total_tokens': 213,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EV0HkchN3tGB3pBLRNUdpsIUNoQeo',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1035f-ca30-7f30-88c3-f34515adfa2e-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK12345'},
                    'id': 'call_PpRt65mwTxedRLz8ZMuGaaVq',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 123,
                'output_tokens': 90,
                'total_tokens': 213,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content=[
                {
                    'type': 'text',
                    'text': 'Booking BK12345 has been cancelled.',
                    'id': 'lc_7a37ce49-b07a-4fc2-91f4-bbfaedffc9d5'
                }
            ],
            name='cancel_booking',
            id='cf676d59-8d32-4c3e-84fa-bd587486aa59',
            tool_call_id='call_PpRt65mwTxedRLz8ZMuGaaVq',
            artifact={'structured_content': {'result': 'Booking BK12345 has been cancelled.'}}
        ),
        AIMessage(
            content='Done — booking BK12345 has been cancelled. Would you like a confirmation email or a refund 
processed?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 162,
                    'total_tokens': 186,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EV0HmxAg0w0QtE72ix1LVZUKU6cHk',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a1035f-d2c8-7ff0-9b38-580ac3ced80a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 162,
                'output_tokens': 24,
                'total_tokens': 186,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0

## connecting to multiple servers 

In [4]:
#MCPConfig-for one config to multiple servers

from fastmcp.mcp_config import MCPConfig

fleet_config: MCPConfig = {
    "mcpServers": {
        "cinebot": {
            "command": "python",
            "args": ["/Users/amohan/personal/anj/personal-projects/agent-projects/agentic-ai-learning-github/agentic-ai-learning-github/krishnaik-agentic-ai-3-0/MCP in Langchain/cinebot_mcp_server.py"],
        },
        "errors": {
            "command": "python",
            "args": ["/Users/amohan/personal/anj/personal-projects/agent-projects/agentic-ai-learning-github/agentic-ai-learning-github/krishnaik-agentic-ai-3-0/MCP in Langchain/cinebot_error_server.py"],
        },
    }
}


async with MCPAdapter(fleet_config) as adapter:
    fleet_tools = await adapter.list_tools()
    print("MCPConfig fleet tools:", [t.name for t in fleet_tools])
    


[10/03/26 16:54:49] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12679683;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12679684;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/03/26 16:54:51] INFO     Proxy detected connected client - reusing existing session for all       ]8;id=12679689;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py\proxy.py]8;;\:]8;id=12679690;file:///Users/amohan/personal/anj/personal-projects/agent-projects/krishnaik-agentic-3.0/mayank-code/.venv/lib/python3.12/site-packages/fastmcp/server/providers/proxy.py#1383\1383]8;;\
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

MCPConfig fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup']


In [5]:
#Different config for different servers
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup

group = ClientGroup(
    {
        "cinebot": Client(Path("../MCP in Langchain/cinebot_mcp_server.py")),
        "errors": Client(Path("../MCP in Langchain/cinebot_error_server.py"),mode="legacy"),
        'time_track_server':Client('https://time-tracker-livid-theta.vercel.app/mcp')
    }
)
async with MCPAdapter(group) as adapter:
    group_tools = await adapter.list_tools()
    print("ClientGroup fleet tools:", [t.name for t in group_tools])

ClientGroup fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup', 'time_track_server_log_time', 'time_track_server_get_timesheet', 'time_track_server_get_project_summary', 'time_track_server_list_projects']


## Connect to MCP server which has authentication

In [12]:
## have MCP server started in the local first as HTTP

async def load_tools_with_bearer(url: str, token: str):
    # `auth` accepts a bearer-token string, the literal "oauth", or any httpx.Auth.
    async with MCPAdapter(Client(url, auth=token)) as adapter:
        return await adapter.list_tools()

# Must match AUTH_TOKEN in cinebot_mcp_server.py
CINEBOT_AUTH_TOKEN = "cinebot-secret-123"

# cinebot_mcp_server.py now serves over HTTP (port 8000) since the bearer
# token check only applies at a network boundary. Start it first with:
#   python cinebot_mcp_server.py

print('connect with valid auth token')
cinebot_tools = await load_tools_with_bearer("http://127.0.0.1:8000/mcp", CINEBOT_AUTH_TOKEN)
print("Tools over HTTP with bearer auth:", [t.name for t in cinebot_tools])


try:
    print('Now, trying connecting with wrong auth token')
    # Must match AUTH_TOKEN in cinebot_mcp_server.py
    CINEBOT_AUTH_TOKEN = "cinebot-secret-123444"
    
    # cinebot_mcp_server.py now serves over HTTP (port 8000) since the bearer
    # token check only applies at a network boundary. Start it first with:
    #   python cinebot_mcp_server.py
    cinebot_tools = await load_tools_with_bearer("http://127.0.0.1:8000/mcp", CINEBOT_AUTH_TOKEN)
    print("Tools over HTTP with bearer auth:", [t.name for t in cinebot_tools])
except Exception as e:
    print(e)

connect with valid auth token
Tools over HTTP with bearer auth: ['check_showtimes', 'cancel_booking', 'get_seat_map']
Now, trying connecting with wrong auth token
Server returned an error response
